In [1]:
from google.colab import drive
drive.mount('/content/drive')

OUTPUT_ROOT = "/content/drive/MyDrive/STAT5398_A2/outputs"


Mounted at /content/drive


In [ ]:
!pip install -q rouge-score nltk bert-score


In [2]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab', quiet=True)  # 有的环境需要这个


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [ ]:
!pip install -U \
  "transformers==4.57.1" \
  "accelerate==0.34.2" \
  "peft==0.13.2" \
  "datasets==2.21.0" \
  "bitsandbytes" \
  "wandb"


In [ ]:
!pip install -U "accelerate>=1.0.1"


  Using cached accelerate-1.11.0-py3-none-any.whl.metadata (19 kB)
Using cached accelerate-1.11.0-py3-none-any.whl (375 kB)
  Attempting uninstall: accelerate
    Found existing installation: accelerate 0.34.2
    Uninstalling accelerate-0.34.2:
      Successfully uninstalled accelerate-0.34.2


In [3]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, TaskType
import wandb, transformers, torch
import os


In [ ]:
  # --- WandB login (run once per runtime) ---
wandb.login()      # 在输出框里粘贴 a9bf8a... 这个 key

# --- HuggingFace login (run once per runtime) ---
from huggingface_hub import login
login()            # 在输出框里粘贴你 HF 的 token


wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: zz3257 (zz3257-columbia-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [4]:
from datasets import load_dataset

dataset_name = "FinGPT/fingpt-forecaster-dow30-202305-202405"
raw_datasets = load_dataset(dataset_name)

print(raw_datasets)
print(raw_datasets["train"][0])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/581 [00:00<?, ?B/s]

data/train-00000-of-00001-7c4c80aa07272d(…):   0%|          | 0.00/3.57M [00:00<?, ?B/s]

(…)-00000-of-00001-28531804b005ddc6.parquet:   0%|          | 0.00/925k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1230 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/300 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['prompt', 'answer', 'period', 'label', 'symbol'],
        num_rows: 1230
    })
    test: Dataset({
        features: ['prompt', 'answer', 'period', 'label', 'symbol'],
        num_rows: 300
    })
})
{'prompt': "[INST]<<SYS>>\nYou are a seasoned stock market analyst. Your task is to list the positive developments and potential concerns for companies based on relevant news and basic financials from the past weeks, then provide an analysis and prediction for the companies' stock price movement for the upcoming week. Your answer format should be as follows:\n\n[Positive Developments]:\n1. ...\n\n[Potential Concerns]:\n1. ...\n\n[Prediction & Analysis]\nPrediction: ...\nAnalysis: ...\n\n<</SYS>>\n\n[Company Introduction]:\n\nAmerican Express Co is a leading entity in the Financial Services sector. Incorporated and publicly traded since 1977-05-18, the company has established its reputation as one of the key players in the market. As of

In [ ]:
def merge_prompt_answer(example):
    # Combine prompt and answer into a single text field
    return {"text": example["prompt"] + example["answer"]}

merged_datasets = raw_datasets.map(merge_prompt_answer)

print(merged_datasets["train"][0]["text"][:500])


Map:   0%|          | 0/1230 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

[INST]<<SYS>>
You are a seasoned stock market analyst. Your task is to list the positive developments and potential concerns for companies based on relevant news and basic financials from the past weeks, then provide an analysis and prediction for the companies' stock price movement for the upcoming week. Your answer format should be as follows:

[Positive Developments]:
1. ...

[Potential Concerns]:
1. ...

[Prediction & Analysis]
Prediction: ...
Analysis: ...

<</SYS>>

[Company Introduction]:


In [ ]:
from transformers import BitsAndBytesConfig

def get_bnb_config():
    """
    4-bit quantization config for fitting 8B models on a single GPU.
    """
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )


def load_tokenizer(model_name: str):
    """
    Load tokenizer and set pad token.
    """
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    return tokenizer


def tokenize_dataset(merged_ds, tokenizer, max_length: int = 2048):
    """
    Tokenize the merged dataset (prompt+answer text).
    We keep it simple: labels = input_ids (standard CLM).
    """

    def tokenize_fn(examples):
        return tokenizer(
            examples["text"],
            truncation=True,
            max_length=max_length,
            padding="max_length",
        )

    tokenized = merged_ds.map(
        tokenize_fn,
        batched=True,
        remove_columns=merged_ds["train"].column_names,
    )

    tokenized.set_format(
        type="torch",
        columns=["input_ids", "attention_mask"],
    )
    return tokenized


from peft import LoraConfig, get_peft_model, TaskType
import torch

def build_lora_model(base_model_name: str):
    """
    Load base model in bf16 and attach LoRA adapters.
    Designed for small models (1B~1.5B) on Colab L4.
    """
    model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        torch_dtype=torch.bfloat16,   # 用 bf16 省显存
        device_map="auto",            # 单卡会自动放到 cuda:0
    )

    # 节省显存
    model.gradient_checkpointing_enable()
    model.config.use_cache = False

    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    return model


In [ ]:
import os
from transformers import (
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer,
)

def finetune_lora(
    base_model_name: str,
    run_name: str,
    tokenized_datasets,
    num_epochs: int,
    max_length: int,
    batch_size: int,
    gradient_accumulation_steps: int,
    learning_rate: float,
    output_root: str = "outputs",
):
    """
    Fine-tune a causal LM with LoRA on the given tokenized dataset.

    Parameters
    ----------
    base_model_name : str
        HF model id, e.g. "meta-llama/Meta-Llama-3.1-8B-Instruct".
    run_name : str
        Name used for output directory and W&B run.
    tokenized_datasets :
        A DatasetDict with splits "train" and "test".
        We will use "train" for training and "test" as validation.
    num_epochs : int
        Number of training epochs.
    max_length : int
        Maximum sequence length (already used in tokenization).
    batch_size : int
        Per-device batch size.
    gradient_accumulation_steps : int
        Gradient accumulation steps to simulate larger batch size.
    learning_rate : float
        Learning rate for AdamW.
    output_root : str
        Root folder to save checkpoints and LoRA adapters.
    """

    import os
    import torch
    from transformers import (
        AutoTokenizer,
        AutoModelForCausalLM,
        DataCollatorForLanguageModeling,
        TrainingArguments,
        Trainer,
    )
    from peft import LoraConfig, get_peft_model, TaskType

    # ---------- 1. Load tokenizer ----------
    tokenizer = AutoTokenizer.from_pretrained(base_model_name, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # ---------- 2. Load base model (no 4-bit to avoid environment issues) ----------
    device_map = "auto" if torch.cuda.is_available() else None
    model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map=device_map,
    )

    # ---------- 3. Attach LoRA adapter ----------
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    # ---------- 4. Data collator ----------
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False,
    )

    # ---------- 5. Choose train / eval splits ----------
    train_dataset = tokenized_datasets["train"]
    # Use "test" split as validation during training
    eval_dataset = tokenized_datasets["test"]

    # ---------- 6. Training arguments ----------
    output_dir = os.path.join(output_root, run_name)
    os.makedirs(output_dir, exist_ok=True)

    training_args = TrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=gradient_accumulation_steps,
        num_train_epochs=num_epochs,
        learning_rate=learning_rate,
        warmup_ratio=0.03,
        logging_steps=50,
        save_steps=200,           # you can adjust this
        save_total_limit=1,
        bf16=torch.cuda.is_available(),  # use bf16 on GPU if available
        report_to=["wandb"],      # log to Weights & Biases
        run_name=run_name,
    )
    # ⚠️ 注意：这里**没有** evaluation_strategy，避免你现在的版本报错

    # ---------- 7. Trainer ----------
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=data_collator,
        tokenizer=tokenizer,
    )

    # ---------- 8. Train ----------
    trainer.train()

    # Optional: run one evaluation at the end
    eval_metrics = trainer.evaluate()
    print("Final eval metrics:", eval_metrics)

    # ---------- 9. Save LoRA adapter ----------
    lora_output_dir = os.path.join(output_dir, "lora_adapter")
    os.makedirs(lora_output_dir, exist_ok=True)

    # Save only LoRA weights
    model.save_pretrained(lora_output_dir)
    tokenizer.save_pretrained(lora_output_dir)

    print("LoRA adapter saved to:", lora_output_dir)
    return lora_output_dir



In [ ]:
# Choose a common max sequence length
MAX_LENGTH = 1024

# We only need to do this ONCE
tokenizer_for_tokenizing = load_tokenizer("meta-llama/Meta-Llama-3.1-8B-Instruct")
tokenized_datasets = tokenize_dataset(merged_datasets, tokenizer_for_tokenizing, max_length=MAX_LENGTH)

print(tokenized_datasets)


tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Map:   0%|          | 0/1230 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 1230
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 300
    })
})


In [ ]:
llama3_base_model = "meta-llama/Llama-3.2-1B-Instruct"
llama3_run_name = "dow30_llama3_1b_lora"

llama3_adapter_dir = finetune_lora(
    base_model_name=llama3_base_model,
    run_name=llama3_run_name,
    tokenized_datasets=tokenized_datasets,
    num_epochs=1,                 # 时间够可以改成 2-3
    max_length=MAX_LENGTH,
    batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=5e-5,
    output_root=OUTPUT_ROOT,
)


tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


...

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

trainable params: 1,703,936 || all params: 1,237,518,336 || trainable%: 0.1377


/tmp/ipython-input-40038259.py:113: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Step,Training Loss
50,2.333000


Final eval metrics: {'eval_loss': 2.1123063564300537, 'eval_runtime': 26.6144, 'eval_samples_per_second': 11.272, 'eval_steps_per_second': 1.428, 'epoch': 1.0}
LoRA adapter saved to: /content/drive/MyDrive/STAT5398_A2/outputs/dow30_llama3_1b_lora/lora_adapter


In [ ]:
deepseek_base_model = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
deepseek_run_name   = "dow30_deepseek_1p5b_lora"

In [ ]:
from transformers import AutoTokenizer

# ==== DeepSeek tokenizer & dataset ====
deepseek_tokenizer = AutoTokenizer.from_pretrained(
    deepseek_base_model,
    trust_remote_code=True,
)

if deepseek_tokenizer.pad_token is None:
    deepseek_tokenizer.pad_token = deepseek_tokenizer.eos_token

def tokenize_for_deepseek(example):
    # Convert our "prompt" and "answer" to a single text sequence
    full_text = example["prompt"] + "\n" + example["answer"]
    return deepseek_tokenizer(
        full_text,
        max_length=MAX_LENGTH,
        truncation=True,
        padding="max_length",
    )

tokenized_datasets_deepseek = raw_datasets.map(
    tokenize_for_deepseek,
    batched=False,
    remove_columns=raw_datasets["train"].column_names,
)

print(tokenized_datasets_deepseek)


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/1230 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 1230
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 300
    })
})


In [ ]:
deepseek_adapter_dir = finetune_lora(
    base_model_name=deepseek_base_model,
    run_name=deepseek_run_name,
    tokenized_datasets=tokenized_datasets_deepseek,
    num_epochs=1,              # 时间允许可以改成 2–3
    max_length=MAX_LENGTH,
    batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=5e-5,
    output_root=OUTPUT_ROOT,
)

print("DeepSeek LoRA adapter dir:", deepseek_adapter_dir)

config.json:   0%|          | 0.00/679 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.55G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

/tmp/ipython-input-40038259.py:113: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 151646, 'pad_token_id': 151643}.


trainable params: 2,179,072 || all params: 1,779,267,072 || trainable%: 0.1225


Step,Training Loss
50,2.840800


Final eval metrics: {'eval_loss': 2.68265700340271, 'eval_runtime': 33.8944, 'eval_samples_per_second': 8.851, 'eval_steps_per_second': 1.121, 'epoch': 1.0}
LoRA adapter saved to: /content/drive/MyDrive/STAT5398_A2/outputs/dow30_deepseek_1p5b_lora/lora_adapter
DeepSeek LoRA adapter dir: /content/drive/MyDrive/STAT5398_A2/outputs/dow30_deepseek_1p5b_lora/lora_adapter


In [8]:
%%writefile comparison.py
# 后面紧跟上面那整段脚本

#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
comparison.py

Evaluate base vs fine-tuned models (Llama3-1B & DeepSeek-1.5B) on the
Dow30 forecasting dataset, and compare their performance.

This script is self-contained:
- does NOT depend on utils.py
- loads models in 4-bit to save GPU memory
- evaluates base model and LoRA fine-tuned model separately
- saves all metrics & inference times under ./comparison_results/
"""

import os
import re
import time
import pickle
from typing import List, Tuple, Dict

import numpy as np
from tqdm import tqdm

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import PeftModel
from datasets import load_dataset, Dataset
from rouge_score import rouge_scorer

# -----------------------
# 1. Global config
# -----------------------

# Hugging Face dataset name (same as training)
DATASET_NAME = "FinGPT/fingpt-forecaster-dow30-202305-202405"

# Paths to your saved LoRA adapters on Google Drive
OUTPUT_ROOT = "/content/drive/MyDrive/STAT5398_A2/outputs"

LLAMA3_BASE_NAME = "meta-llama/Llama-3.2-1B-Instruct"
LLAMA3_ADAPTER_DIR = os.path.join(
    OUTPUT_ROOT, "dow30_llama3_1b_lora", "lora_adapter"
)

DEEPSEEK_BASE_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
DEEPSEEK_ADAPTER_DIR = os.path.join(
    OUTPUT_ROOT, "dow30_deepseek_1p5b_lora", "lora_adapter"
)

RESULT_DIR = "/content/drive/MyDrive/STAT5398_A2/comparison_results"
os.makedirs(RESULT_DIR, exist_ok=True)

MAX_INPUT_LEN = 2048
MAX_NEW_TOKENS = 512

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# 4-bit quantization config (same style as training)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

# Rouge scorer
rouge = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)


# -----------------------
# 2. Dataset helpers
# -----------------------

INTRO_MARKER = (
    "[INST]<<SYS>>\n"
    "You are a seasoned stock market analyst. Your task is to list the positive developments and "
    "potential concerns for companies based on relevant news and basic financials from the past weeks, "
    "then provide an analysis and prediction for the companies' stock price movement for the upcoming week."
)
GUIDANCE_START = "Based on all the information before"
GUIDANCE_END = "Following these instructions, please come up with 2-4 most important positive factors"


def insert_guidance_after_intro(prompt: str) -> str:
    """
    Reorder the guidance section so that it appears right after the intro system message.
    This follows the instructor's comparison.py logic.
    """
    intro_pos = prompt.find(INTRO_MARKER)
    g_start = prompt.find(GUIDANCE_START)
    g_end = prompt.find(GUIDANCE_END)

    if intro_pos == -1 or g_start == -1 or g_end == -1:
        # If we cannot find all markers, return original prompt
        return prompt

    guidance_section = prompt[g_start:g_end].strip()

    new_prompt = (
        f"{prompt[:intro_pos + len(INTRO_MARKER)]}\n\n"
        f"{guidance_section}\n\n"
        f"{prompt[intro_pos + len(INTRO_MARKER):g_start]}"
        f"{prompt[g_end:]}"
    )
    return new_prompt


def apply_to_all_prompts(test_dataset: Dataset) -> Dataset:
    """Apply guidance reordering to all prompts in the dataset."""
    return test_dataset.map(
        lambda x: {"prompt": insert_guidance_after_intro(x["prompt"])},
        desc="Rewriting prompts with guidance section",
    )


# -----------------------
# 3. Metrics
# -----------------------

def calc_metrics(preds: List[str], refs: List[str]) -> Dict[str, float]:
    """
    Compute text generation metrics between predictions and references.
    - Rouge-1 F1
    - Rouge-L F1
    - Exact match rate (strict string equality after strip)
    """
    rouge1_f, rougeL_f, em = [], [], []

    for p, r in zip(preds, refs):
        p = p.strip()
        r = r.strip()
        scores = rouge.score(r, p)  # ref, hyp
        rouge1_f.append(scores["rouge1"].fmeasure)
        rougeL_f.append(scores["rougeL"].fmeasure)
        em.append(float(p == r))

    return {
        "rouge1_f": float(np.mean(rouge1_f)) if rouge1_f else 0.0,
        "rougeL_f": float(np.mean(rougeL_f)) if rougeL_f else 0.0,
        "exact_match": float(np.mean(em)) if em else 0.0,
    }


# -----------------------
# 4. Model loading & generation
# -----------------------

def load_model_and_tokenizer(
    base_model_name: str,
    adapter_dir: str = None,
):
    """
    Load base model in 4-bit and optionally attach a LoRA adapter.
    We use device_map='auto' so that HF dispatches weights to GPU.
    """
    print(f"[INFO] Loading base model: {base_model_name}")
    model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )

    tokenizer = AutoTokenizer.from_pretrained(
        base_model_name,
        use_fast=True,
    )
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id
    tokenizer.padding_side = "right"

    if adapter_dir is not None:
        print(f"[INFO] Attaching LoRA adapter from: {adapter_dir}")
        model = PeftModel.from_pretrained(
            model,
            adapter_dir,
        )

    model.eval()
    return model, tokenizer


def extract_answer(output_text: str) -> str:
    """
    Post-process the raw generation:
    - keep only content after '[/INST]'
    - strip leading/trailing whitespace
    """
    m = re.search(r"\[/INST\]\s*(.*)", output_text, flags=re.DOTALL)
    if m:
        return m.group(1).strip()
    return output_text.strip()


def generate_one(
    model,
    tokenizer,
    prompt: str,
    max_input_len: int = MAX_INPUT_LEN,
    max_new_tokens: int = MAX_NEW_TOKENS,
) -> Tuple[str, float]:
    """Generate one answer and measure latency."""
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_len,
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    start = time.time()
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )
    end = time.time()

    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    return extract_answer(decoded), (end - start)


def run_eval_loop(
    model,
    tokenizer,
    test_dataset: Dataset,
    desc: str,
) -> Tuple[List[str], List[str], List[float]]:
    """Run generation for every sample in the test set."""
    preds, refs, times = [], [], []

    for row in tqdm(test_dataset, desc=desc):
        prompt = row["prompt"]
        gt = row["answer"]

        ans, t = generate_one(model, tokenizer, prompt)
        preds.append(ans)
        refs.append(gt)
        times.append(t)

    return preds, refs, times


# -----------------------
# 5. High-level evaluation for one model family
# -----------------------

def evaluate_model_family(
    base_model_name: str,
    adapter_dir: str,
    test_dataset: Dataset,
    tag: str,
):
    """
    Evaluate (1) base model and (2) LoRA fine-tuned model for the same family.

    Returns a dict containing:
        - base_metrics, ft_metrics
        - base_times, ft_times
        - base_answers, ft_answers, gts
    """
    # 5.1 Base model
    base_model, tokenizer = load_model_and_tokenizer(base_model_name, adapter_dir=None)
    base_answers, gts, base_times = run_eval_loop(
        base_model,
        tokenizer,
        test_dataset,
        desc=f"{tag} | base model",
    )
    base_metrics = calc_metrics(base_answers, gts)
    print(f"[RESULT] {tag} base metrics:", base_metrics)

    # free memory
    del base_model
    torch.cuda.empty_cache()

    # 5.2 Fine-tuned model
    ft_model, tokenizer = load_model_and_tokenizer(base_model_name, adapter_dir=adapter_dir)
    ft_answers, gts2, ft_times = run_eval_loop(
        ft_model,
        tokenizer,
        test_dataset,
        desc=f"{tag} | fine-tuned",
    )
    assert gts == gts2  # sanity check
    ft_metrics = calc_metrics(ft_answers, gts)
    print(f"[RESULT] {tag} fine-tuned metrics:", ft_metrics)

    del ft_model
    torch.cuda.empty_cache()

    return {
        "base_metrics": base_metrics,
        "ft_metrics": ft_metrics,
        "base_times": base_times,
        "ft_times": ft_times,
        "base_answers": base_answers,
        "ft_answers": ft_answers,
        "gts": gts,
    }


# -----------------------
# 6. Main
# -----------------------

def main():
    # 6.1 Load dataset and test split
    print(f"[INFO] Loading dataset: {DATASET_NAME}")
    raw = load_dataset(DATASET_NAME)
    test_ds = raw["test"]
    print(f"[INFO] Raw test size: {len(test_ds)}")

    # 6.2 Apply prompt rewriting (guidance after intro)
    test_ds = apply_to_all_prompts(test_ds)
    print(f"[INFO] Test size after rewriting: {len(test_ds)}")

    # 6.3 Evaluate Llama3 family
    llama3_results = evaluate_model_family(
        LLAMA3_BASE_NAME,
        LLAMA3_ADAPTER_DIR,
        test_ds,
        tag="Llama3-1B",
    )

    # Save Llama3 results
    with open(os.path.join(RESULT_DIR, "llama3_base_metrics.pkl"), "wb") as f:
        pickle.dump(llama3_results["base_metrics"], f)

    with open(os.path.join(RESULT_DIR, "llama3_fine_tuned_metrics.pkl"), "wb") as f:
        pickle.dump(llama3_results["ft_metrics"], f)

    with open(os.path.join(RESULT_DIR, "llama3_base_times.pkl"), "wb") as f:
        pickle.dump(llama3_results["base_times"], f)

    with open(os.path.join(RESULT_DIR, "llama3_fine_tuned_times.pkl"), "wb") as f:
        pickle.dump(llama3_results["ft_times"], f)

    # 6.4 Evaluate DeepSeek family
    deepseek_results = evaluate_model_family(
        DEEPSEEK_BASE_NAME,
        DEEPSEEK_ADAPTER_DIR,
        test_ds,
        tag="DeepSeek-1.5B",
    )

    with open(os.path.join(RESULT_DIR, "deepseek_base_metrics.pkl"), "wb") as f:
        pickle.dump(deepseek_results["base_metrics"], f)

    with open(os.path.join(RESULT_DIR, "deepseek_fine_tuned_metrics.pkl"), "wb") as f:
        pickle.dump(deepseek_results["ft_metrics"], f)

    with open(os.path.join(RESULT_DIR, "deepseek_base_times.pkl"), "wb") as f:
        pickle.dump(deepseek_results["base_times"], f)

    with open(os.path.join(RESULT_DIR, "deepseek_fine_tuned_times.pkl"), "wb") as f:
        pickle.dump(deepseek_results["ft_times"], f)

    # 6.5 Cross-model comparison (fine-tuned Llama3 vs fine-tuned DeepSeek)
    print("\n[INFO] Comparing fine-tuned Llama3 vs fine-tuned DeepSeek...")
    comparison_metrics = calc_metrics(
        llama3_results["ft_answers"],
        deepseek_results["ft_answers"],
    )

    with open(os.path.join(RESULT_DIR, "comparison_metrics.pkl"), "wb") as f:
        pickle.dump(comparison_metrics, f)

    print("\n[RESULT] Cross-model comparison metrics:", comparison_metrics)
    print(f"\nAll evaluation finished. Results saved under {RESULT_DIR}/")


if __name__ == "__main__":
    main()


Writing comparison.py


In [ ]:
!python comparison.py


2025-11-21 05:37:50.930558: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763703470.954058    4846 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763703470.961211    4846 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1763703470.979055    4846 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1763703470.979085    4846 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1763703470.979088    4846 computation_placer.cc:177] computation placer alr

In [ ]:
!ls comparison_results

In [5]:
import os, pickle

RESULT_DIR = "/content/drive/MyDrive/STAT5398_A2/comparison_results"
files = os.listdir(RESULT_DIR)
print(files)

for name in files:
    path = os.path.join(RESULT_DIR, name)
    with open(path, "rb") as f:
        obj = pickle.load(f)
    print("\n>>>", name)
    print(obj)


['llama3_base_metrics.pkl', 'llama3_fine_tuned_metrics.pkl', 'llama3_base_times.pkl', 'llama3_fine_tuned_times.pkl', 'deepseek_base_metrics.pkl', 'deepseek_fine_tuned_metrics.pkl', 'deepseek_base_times.pkl', 'deepseek_fine_tuned_times.pkl', 'comparison_metrics.pkl']

>>> llama3_base_metrics.pkl
{'rouge1_f': 0.3703813140392724, 'rougeL_f': 0.17688079351852598, 'exact_match': 0.0}

>>> llama3_fine_tuned_metrics.pkl
{'rouge1_f': 0.34615756294496675, 'rougeL_f': 0.16446655137223265, 'exact_match': 0.0}

>>> llama3_base_times.pkl
[1.706350564956665, 20.535805702209473, 20.021857738494873, 20.216025352478027, 14.850709438323975, 19.96190619468689, 18.375584840774536, 20.110966205596924, 0.07509684562683105, 17.030375719070435, 15.604492664337158, 20.430872917175293, 20.23889946937561, 20.24813222885132, 20.353706121444702, 20.2081401348114, 20.209420204162598, 13.543615102767944, 20.23580813407898, 15.234777688980103, 20.27885913848877, 20.32674503326416, 18.78165555000305, 20.23555302619934

In [6]:
import pandas as pd
import numpy as np
import pickle
import os

RESULT_DIR = "/content/drive/MyDrive/STAT5398_A2/comparison_results"

def load(name):
    with open(os.path.join(RESULT_DIR, name), "rb") as f:
        return pickle.load(f)

llama3_base = load("llama3_base_metrics.pkl")
llama3_ft   = load("llama3_fine_tuned_metrics.pkl")
deep_base   = load("deepseek_base_metrics.pkl")
deep_ft     = load("deepseek_fine_tuned_metrics.pkl")
comp_ft     = load("comparison_metrics.pkl")

df = pd.DataFrame([
    ["Llama3-1B base",      llama3_base["rouge1_f"], llama3_base["rougeL_f"], llama3_base["exact_match"]],
    ["Llama3-1B fine-tune", llama3_ft["rouge1_f"],   llama3_ft["rougeL_f"],   llama3_ft["exact_match"]],
    ["DeepSeek-1.5B base",  deep_base["rouge1_f"],   deep_base["rougeL_f"],   deep_base["exact_match"]],
    ["DeepSeek-1.5B ft",    deep_ft["rouge1_f"],     deep_ft["rougeL_f"],     deep_ft["exact_match"]],
], columns=["Model", "ROUGE-1 F", "ROUGE-L F", "Exact Match"])

df


,Model,ROUGE-1 F,ROUGE-L F,Exact Match
0,Llama3-1B base,0.370381,0.176881,0.0
1,Llama3-1B fine-tune,0.346158,0.164467,0.0
2,DeepSeek-1.5B base,0.378596,0.162732,0.0
3,DeepSeek-1.5B ft,0.380051,0.168016,0.0


In [7]:
import numpy as np

llama3_base_t = np.mean(load("llama3_base_times.pkl"))
llama3_ft_t   = np.mean(load("llama3_fine_tuned_times.pkl"))
deep_base_t   = np.mean(load("deepseek_base_times.pkl"))
deep_ft_t     = np.mean(load("deepseek_fine_tuned_times.pkl"))

print("Avg latency (sec / sample):")
print("Llama3 base    :", llama3_base_t)
print("Llama3 fine-tune:", llama3_ft_t)
print("DeepSeek base  :", deep_base_t)
print("DeepSeek fine-tune:", deep_ft_t)


Avg latency (sec / sample):
Llama3 base    : 18.34733945608139
Llama3 fine-tune: 28.358007764816286
DeepSeek base  : 29.655250799655914
DeepSeek fine-tune: 41.570848228931425


In [9]:
!cp /content/comparison.py /content/drive/MyDrive/STAT5398_A2/
